# 03 — Quejas y abandono (H3.3)
Hipótesis: los clientes con quejas no resueltas dejan de comprar más que los que vieron su queja resuelta.
Fuente: data/clean/ (salida del notebook 01).

In [1]:
import pandas as pd
from pathlib import Path
from statsmodels.stats.proportion import proportions_ztest

CLEAN = Path("../data/clean")
crm = pd.read_csv(CLEAN / "interacciones_crm.csv", parse_dates=["fecha"])
tx = pd.read_csv(CLEAN / "transacciones.csv", parse_dates=["fecha"])

quejas = crm[crm.tipo == "queja"]
fin = tx.fecha.max()
print(f"Quejas: {len(quejas):,} | Fin de datos: {fin.date()}")
quejas.resolucion.value_counts()

Quejas: 2,081 | Fin de datos: 2024-12-31


resolucion
resuelto                 805
sin_resolver             499
escalado                 471
parcialmente_resuelto    306
Name: count, dtype: int64

## Compras alrededor de cada queja
Para cada ventana N (30, 90 y 180 días) solo se cuentan las quejas con N días de datos por delante.
- **Compró después**: alguna compra del mismo cliente entre 1 y N días después de la queja.
- **Gasto antes / después**: suma de monto_neto en los N días previos / siguientes.

In [2]:
cruce = quejas[["interaccion_id", "cliente_id", "fecha", "resolucion"]].merge(
    tx[["cliente_id", "fecha", "monto_neto"]].rename(columns={"fecha": "fecha_compra"}), on="cliente_id", how="left")
cruce["dias"] = (cruce.fecha_compra - cruce.fecha).dt.days

por_ventana = {}
for N in [30, 90, 180]:
    q = quejas[quejas.fecha <= fin - pd.Timedelta(days=N)].set_index("interaccion_id")[["resolucion"]].copy()
    c = cruce[cruce.interaccion_id.isin(q.index)]
    despues = c[(c.dias > 0) & (c.dias <= N)].groupby("interaccion_id").monto_neto.sum()
    antes = c[(c.dias >= -N) & (c.dias < 0)].groupby("interaccion_id").monto_neto.sum()
    q["compra_despues"] = q.index.isin(despues.index)
    q["gasto_despues"] = despues.reindex(q.index).fillna(0)
    q["gasto_antes"] = antes.reindex(q.index).fillna(0)
    por_ventana[N] = q

tabla = pd.concat({N: q.groupby("resolucion").agg(
    quejas=("compra_despues", "size"), pct_compra_despues=("compra_despues", lambda s: s.mean() * 100),
    gasto_antes=("gasto_antes", "mean"), gasto_despues=("gasto_despues", "mean"))
    for N, q in por_ventana.items()}, names=["ventana_dias"])
tabla["cambio"] = tabla.gasto_despues - tabla.gasto_antes
tabla.round(2)

quejas  pct_compra_despues  gasto_antes  \
ventana_dias resolucion                                                       
30           escalado                  449               21.38        92.99   
             parcialmente_resuelto     294               18.03       122.97   
             resuelto                  776               23.71        99.97   
             sin_resolver              481               21.00        80.87   
90           escalado                  405               47.90       296.00   
             parcialmente_resuelto     275               48.36       303.23   
             resuelto                  713               50.07       274.02   
             sin_resolver              443               46.95       303.04   
180          escalado                  340               64.71       587.77   
             parcialmente_resuelto     238               65.97       480.09   
             resuelto                  624               67.95       491.04   
             sin_resolver              369               63.14       587.09   

                                    gasto_despues  cambio  
ventana_dias resolucion                                    
30           escalado                       92.99    0.00  
             parcialmente_resuelto          69.25  -53.72  
             resuelto                      118.27   18.30  
             sin_resolver                   84.90    4.03  
90           escalado                      342.13   46.13  
             parcialmente_resuelto         265.70  -37.52  
             resuelto                      337.70   63.68  
             sin_resolver                  304.44    1.40  
180          escalado                      625.73   37.96  
             parcialmente_resuelto         565.50   85.41  
             resuelto                      679.70  188.66  
             sin_resolver                  645.04   57.95

In [3]:
# A 90 días, solo clientes que ya compraban en los 90 días previos a la queja
q90 = por_ventana[90]
(q90[q90.gasto_antes > 0].groupby("resolucion").compra_despues.mean() * 100).round(1).rename("pct_sigue_comprando")

resolucion
escalado                 64.9
parcialmente_resuelto    58.3
resuelto                 65.3
sin_resolver             65.4
Name: pct_sigue_comprando, dtype: float64

In [4]:
# Diferencia de proporciones: resuelto contra sin_resolver
pruebas = []
for N, q in por_ventana.items():
    g = q[q.resolucion.isin(["resuelto", "sin_resolver"])].groupby("resolucion").compra_despues.agg(["sum", "size"])
    z, p = proportions_ztest(g["sum"].values, g["size"].values)
    pruebas.append({"ventana_dias": N,
                    "resuelto_pct": g.loc["resuelto", "sum"] / g.loc["resuelto", "size"] * 100,
                    "sin_resolver_pct": g.loc["sin_resolver", "sum"] / g.loc["sin_resolver", "size"] * 100,
                    "z": z, "p_valor": p})
pruebas = pd.DataFrame(pruebas).set_index("ventana_dias")
pruebas["diferencia_pts"] = pruebas.resuelto_pct - pruebas.sin_resolver_pct
pruebas.round(3)

,resuelto_pct,sin_resolver_pct,z,p_valor,diferencia_pts
ventana_dias,,,,,
30,23.711,20.998,1.117,0.264,2.713
90,50.070,46.953,1.031,0.303,3.118
180,67.949,63.144,1.546,0.122,4.805


**H3.3 no concluyente: efecto de 3 a 5 puntos, no significativo.**
- Los clientes con la queja resuelta vuelven a comprar algo más que los que no: 23,7% contra 21,0%
  a 30 días, 50,1% contra 47,0% a 90 días y 67,9% contra 63,1% a 180 días.
- La diferencia no es estadísticamente significativa en ninguna ventana (p > 0,05).
- Entre quienes ya compraban antes de la queja no hay diferencia: 65,3% contra 65,4% a 90 días.
- A 180 días el gasto sube en ambos grupos (+$188,66 resuelto, +$57,95 sin resolver).
  Con estos datos no se puede afirmar que resolver la queja retenga clientes.